# Bellabeat Case Study: How Can a Wellness Company Play It Smart?

Google Data Analytics Capstone, Case Study 2

Tools: Python (pandas, seaborn, matplotlib) and SQL (SQLite) in a Jupyter notebook

The notebook follows the six phases of the analysis process: Ask, Prepare, Process, Analyze, Share and Act. The charts are also saved as PNG files in a `visuals` folder so they can be reused in a presentation or in Tableau.

## 1. Ask

**Business task**

Bellabeat is a small but fast growing maker of health focused smart products for women. Urska Srsen, the Chief Creative Officer, believes that smart device usage data can point to new growth opportunities. My job is to study how people use non Bellabeat smart devices, then apply what I learn to one Bellabeat product and turn it into marketing recommendations.

**Guiding questions**

1. What are some trends in smart device usage?
2. How could these trends apply to Bellabeat customers?
3. How could these trends help influence Bellabeat marketing strategy?

**Stakeholders**

* Urska Srsen, cofounder and Chief Creative Officer
* Sando Mur, cofounder and member of the executive team
* The Bellabeat marketing analytics team

**Deliverable for this phase:** one clear statement of the task. *Find the usage trends in Fitbit data that can guide the marketing strategy for one Bellabeat product.*

## 2. Prepare

**Data source:** FitBit Fitness Tracker Data (Kaggle, made available through Mobius, CC0 public domain). It holds personal tracker data from Fitbit users who agreed to share it, collected through a survey on Amazon Mechanical Turk between 03.12.2016 and 05.12.2016. The download comes as two folders, one per collection period. The files are in long format, with one row per user and per day, hour or minute.

**Files used**

| File | Folder | Why I use it |
|---|---|---|
| dailyActivity_merged | both | steps, distance, intensity minutes and calories per day |
| hourlySteps_merged | both | when during the day people move |
| sleepDay_merged | 4.12.16 to 5.12.16 | sleep minutes and time in bed per night |
| weightLogInfo_merged | 4.12.16 to 5.12.16 | how many people log weight and how |

The minute level files (calories, intensities, METs, steps, sleep) and the heart rate file are very large, and they repeat what the daily and hourly files already show, so I left them out.

**Does the data ROCCC?**

* **Reliable:** low. Only about thirty people, and the sample is not random.
* **Original:** third party data collected through a survey, not first party Bellabeat data.
* **Comprehensive:** partial. Activity, sleep and weight are covered, but there is nothing on stress, menstrual cycle or hydration, which are core Bellabeat features.
* **Current:** weak. The data is from spring 2016, so habits may have changed.
* **Cited:** yes, the source and the license are clear.

**Limitations to keep in mind**

* Small sample, so results describe this group and may not generalize.
* About one month per folder, so long term habits and seasonality cannot be studied.
* No demographic columns. I cannot confirm that the users are women, which is Bellabeat's audience.
* Users who share data are probably more engaged with fitness than the average buyer.

**Privacy and licensing:** the data is anonymized (numeric IDs only) and released as public domain, so it can be used freely. No attempt is made to identify anyone.

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

os.makedirs("visuals", exist_ok=True)

: 

In [ ]:
early = "Fitabase Data 3.12.16-4.11.16"
late = "Fitabase Data 4.12.16-5.12.16"

def read(folder, name):
    return pd.read_csv(os.path.join(folder, name))

daily_early = read(early, "dailyActivity_merged.csv")
daily_late = read(late, "dailyActivity_merged.csv")
hourly_early = read(early, "hourlySteps_merged.csv")
hourly_late = read(late, "hourlySteps_merged.csv")
sleep_raw = read(late, "sleepDay_merged.csv")
weight_raw = read(late, "weightLogInfo_merged.csv")

raw_tables = {
    "daily_early": daily_early,
    "daily_late": daily_late,
    "hourly_early": hourly_early,
    "hourly_late": hourly_late,
    "sleep": sleep_raw,
    "weight": weight_raw,
}

overview = pd.DataFrame({
    "rows": {k: len(v) for k, v in raw_tables.items()},
    "columns": {k: v.shape[1] for k, v in raw_tables.items()},
    "unique_users": {k: v["Id"].nunique() for k, v in raw_tables.items()},
    "missing_cells": {k: int(v.isna().sum().sum()) for k, v in raw_tables.items()},
    "duplicate_rows": {k: int(v.duplicated().sum()) for k, v in raw_tables.items()},
})
overview

In [ ]:
daily_late.head()

In [ ]:
daily_late.dtypes

**First look at the data**

* The tables are in long format and share an `Id` column, so they can be joined on user and date.
* Dates are stored as text, so they need to be converted before any weekday or hourly work.
* The user counts per table do not all match the "thirty users" in the description, so I will check that carefully during processing.
* Weight has a lot of missing cells, which I will look at in the next phase.

## 3. Process

**Tools:** Python with pandas for cleaning, because it keeps every step documented and repeatable in one notebook, and SQLite for the analysis queries, which also gives me a SQL version of the key results.

**Cleaning plan**

1. Convert IDs to text, since they are labels and not numbers.
2. Convert date columns to proper datetimes.
3. Stack the two collection periods of daily and hourly data, then remove duplicates.
4. Remove exact duplicate rows in the sleep data.
5. Look at missing values and decide what to keep.
6. Find days where the device was probably not worn and exclude them.
7. Create the new columns needed for the analysis.
8. Run integrity checks.

In [ ]:
# 1 and 2: IDs as text, dates as datetimes
def fix_id(df):
    df = df.copy()
    df["Id"] = df["Id"].astype(str)
    return df

daily = pd.concat([fix_id(daily_early), fix_id(daily_late)], ignore_index=True)
daily["ActivityDate"] = pd.to_datetime(daily["ActivityDate"], format="%m/%d/%Y")

hourly = pd.concat([fix_id(hourly_early), fix_id(hourly_late)], ignore_index=True)
hourly["ActivityHour"] = pd.to_datetime(hourly["ActivityHour"], format="%m/%d/%Y %I:%M:%S %p")

sleep = fix_id(sleep_raw)
sleep["SleepDay"] = pd.to_datetime(sleep["SleepDay"], format="%m/%d/%Y %I:%M:%S %p").dt.normalize()

weight = fix_id(weight_raw)
weight["Date"] = pd.to_datetime(weight["Date"], format="%m/%d/%Y %I:%M:%S %p")

print("Daily dates :", daily["ActivityDate"].min().date(), "to", daily["ActivityDate"].max().date())
print("Hourly dates:", hourly["ActivityHour"].min().date(), "to", hourly["ActivityHour"].max().date())
print("Sleep dates :", sleep["SleepDay"].min().date(), "to", sleep["SleepDay"].max().date())

In [ ]:
# 3 and 4: duplicates
before = {"daily": len(daily), "hourly": len(hourly), "sleep": len(sleep)}

# the two folders share a few days, so look at repeated user and date pairs
daily = daily.drop_duplicates(subset=["Id", "ActivityDate"], keep="last")
hourly = hourly.drop_duplicates(subset=["Id", "ActivityHour"], keep="last")
sleep = sleep.drop_duplicates()

after = {"daily": len(daily), "hourly": len(hourly), "sleep": len(sleep)}
pd.DataFrame({"before": before, "after": after, "removed": {k: before[k] - after[k] for k in before}})

In [ ]:
# 5: missing values
for name, df in {"daily": daily, "hourly": hourly, "sleep": sleep, "weight": weight}.items():
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(name, "->", "no missing values" if missing.empty else dict(missing))

In [ ]:
weight_users = weight["Id"].nunique()
manual = weight["IsManualReport"].value_counts()
print("Users with any weight entry:", weight_users)
print("Total weight entries:", len(weight))
print(manual)

Only the `Fat` column in the weight table has missing values, and it is almost empty, so I drop it. The weight table itself covers very few users, so I will treat it as a feature usage signal and not as a base for conclusions about weight change.

In [ ]:
weight = weight.drop(columns=["Fat"])

# 6: probable non wear days
minute_cols = ["VeryActiveMinutes", "FairlyActiveMinutes", "LightlyActiveMinutes", "SedentaryMinutes"]
daily["MinutesTracked"] = daily[minute_cols].sum(axis=1)

zero_steps = (daily["TotalSteps"] == 0).sum()
full_sedentary = (daily["SedentaryMinutes"] == 1440).sum()
print("Days with zero steps:", zero_steps)
print("Days with 1440 sedentary minutes:", full_sedentary)
print("Days with 1440 tracked minutes:", (daily["MinutesTracked"] == 1440).sum(), "of", len(daily))
daily.loc[daily["TotalSteps"] == 0, ["Id", "ActivityDate", "TotalSteps", "SedentaryMinutes", "Calories"]].head()

A day with zero steps and almost all minutes sedentary most likely means the tracker was not worn or was out of battery, not that the person did nothing. Those days would pull the averages down without telling us anything about behavior, so I exclude them from the activity analysis and keep a record of how many there were.

In [ ]:
daily_all = daily.copy()
daily = daily[daily["TotalSteps"] > 0].copy()
print(f"Removed {len(daily_all) - len(daily)} probable non wear days, {len(daily)} days remain")

# 7: new columns
daily["Weekday"] = pd.Categorical(
    daily["ActivityDate"].dt.day_name(),
    categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    ordered=True,
)
daily["ActiveMinutes"] = daily["VeryActiveMinutes"] + daily["FairlyActiveMinutes"] + daily["LightlyActiveMinutes"]
daily["ModerateToVigorous"] = daily["VeryActiveMinutes"] + daily["FairlyActiveMinutes"]

hourly["Hour"] = hourly["ActivityHour"].dt.hour
hourly["Weekday"] = pd.Categorical(
    hourly["ActivityHour"].dt.day_name(),
    categories=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
    ordered=True,
)

sleep["SleepEfficiency"] = sleep["TotalMinutesAsleep"] / sleep["TotalTimeInBed"]
sleep["HoursAsleep"] = sleep["TotalMinutesAsleep"] / 60
sleep["MinutesAwakeInBed"] = sleep["TotalTimeInBed"] - sleep["TotalMinutesAsleep"]

# daily activity joined with sleep, on user and date
daily_sleep = daily.merge(
    sleep, left_on=["Id", "ActivityDate"], right_on=["Id", "SleepDay"], how="inner"
)
print("Days with both activity and sleep:", len(daily_sleep), "from", daily_sleep["Id"].nunique(), "users")

In [ ]:
# 8: integrity checks
hourly_totals = (
    hourly.assign(day=hourly["ActivityHour"].dt.normalize())
    .groupby(["Id", "day"], as_index=False)["StepTotal"].sum()
)
check = daily.merge(hourly_totals, left_on=["Id", "ActivityDate"], right_on=["Id", "day"])
match_rate = (check["TotalSteps"] == check["StepTotal"]).mean()

checks = pd.Series({
    "no repeated user days in daily": not daily.duplicated(subset=["Id", "ActivityDate"]).any(),
    "no repeated user hours in hourly": not hourly.duplicated(subset=["Id", "ActivityHour"]).any(),
    "steps are positive": bool((daily["TotalSteps"] > 0).all()),
    "calories are positive": bool((daily["Calories"] > 0).all()),
    "tracked minutes within one day": bool((daily["MinutesTracked"] <= 1440).all()),
    "asleep never longer than in bed": bool((sleep["TotalMinutesAsleep"] <= sleep["TotalTimeInBed"]).all()),
    "hours between 0 and 23": bool(hourly["Hour"].between(0, 23).all()),
}, name="passed")
print(checks.to_string())
print(f"Daily and hourly steps match on {match_rate:.1%} of {len(check)} user days")

users_daily = daily["Id"].nunique()
users_sleep = sleep["Id"].nunique()
print("Users in daily activity:", users_daily)
print("Users in sleep:", users_sleep)
print("Users in weight:", weight["Id"].nunique())

**Cleaning log**

* IDs converted to text and all date columns converted to datetimes.
* The two collection periods were stacked for daily and hourly data, and repeated user and date pairs were removed.
* Exact duplicate sleep rows were removed.
* The `Fat` column in the weight table was dropped because it is almost empty.
* Days with zero steps were treated as non wear days and excluded from the activity analysis.
* New columns were created: weekday, hour, active minutes, moderate to vigorous minutes, sleep efficiency, hours asleep and minutes awake in bed.
* Integrity checks were run on repeated user days, minutes within a 24 hour day, sleep versus time in bed, and agreement between hourly and daily steps. The results are printed above.
* The original CSV files were never changed. All edits happen on copies in memory.

## 4. Analyze

I start with descriptive statistics, then look at how people differ from each other, then at time patterns (weekday and hour), and finally at sleep. The key queries are written in SQL so they can be reused outside Python.

In [ ]:
summary_cols = [
    "TotalSteps", "TotalDistance", "VeryActiveMinutes", "FairlyActiveMinutes",
    "LightlyActiveMinutes", "SedentaryMinutes", "Calories",
]
daily[summary_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]]

In [ ]:
sleep[["TotalMinutesAsleep", "TotalTimeInBed", "SleepEfficiency", "MinutesAwakeInBed"]].describe().T[
    ["count", "mean", "std", "min", "50%", "max"]
]

In [ ]:
# load the clean tables into SQLite so the same analysis can be done with SQL
conn = sqlite3.connect(":memory:")

daily_sql = daily.drop(columns=["Weekday"]).copy()
daily_sql["ActivityDate"] = daily_sql["ActivityDate"].dt.strftime("%Y-%m-%d")
daily_sql.to_sql("daily_activity", conn, index=False)

hourly_sql = hourly.drop(columns=["Weekday"]).copy()
hourly_sql["ActivityHour"] = hourly_sql["ActivityHour"].dt.strftime("%Y-%m-%d %H:%M:%S")
hourly_sql.to_sql("hourly_steps", conn, index=False)

sleep_sql = sleep.copy()
sleep_sql["SleepDay"] = sleep_sql["SleepDay"].dt.strftime("%Y-%m-%d")
sleep_sql.to_sql("sleep_day", conn, index=False)

weight_sql = weight.copy()
weight_sql["Date"] = weight_sql["Date"].dt.strftime("%Y-%m-%d %H:%M:%S")
weight_sql.to_sql("weight_log", conn, index=False)

def sql(query):
    return pd.read_sql_query(query, conn)

sql("SELECT name FROM sqlite_master WHERE type = 'table'")

### 4.1 How many people use each feature?

If a feature is rarely used, that is a hint about friction or about what people value.

In [ ]:
feature_usage = sql("""
    SELECT 'Daily activity' AS feature, COUNT(DISTINCT Id) AS users, COUNT(*) AS records FROM daily_activity
    UNION ALL
    SELECT 'Hourly steps', COUNT(DISTINCT Id), COUNT(*) FROM hourly_steps
    UNION ALL
    SELECT 'Sleep', COUNT(DISTINCT Id), COUNT(*) FROM sleep_day
    UNION ALL
    SELECT 'Weight', COUNT(DISTINCT Id), COUNT(*) FROM weight_log
""")
feature_usage["share_of_activity_users"] = feature_usage["users"] / feature_usage.loc[0, "users"]
feature_usage

In [ ]:
weight_manual = sql("""
    SELECT IsManualReport, COUNT(*) AS entries, COUNT(DISTINCT Id) AS users
    FROM weight_log
    GROUP BY IsManualReport
""")
weight_manual

### 4.2 What kind of users are in the data?

I group each person by their average daily steps. The cut points follow a common public health scale for step counts: under 5,000 is sedentary, 5,000 to 7,499 is low active, 7,500 to 9,999 is somewhat active, and 10,000 or more is active.

In [ ]:
user_types = sql("""
    WITH per_user AS (
        SELECT Id,
               COUNT(*)                 AS days_logged,
               AVG(TotalSteps)          AS avg_steps,
               AVG(Calories)            AS avg_calories,
               AVG(SedentaryMinutes)    AS avg_sedentary_min,
               AVG(VeryActiveMinutes + FairlyActiveMinutes) AS avg_mvpa_min
        FROM daily_activity
        GROUP BY Id
    )
    SELECT CASE
               WHEN avg_steps >= 10000 THEN '4 Active (10,000+)'
               WHEN avg_steps >= 7500  THEN '3 Somewhat active (7,500 to 9,999)'
               WHEN avg_steps >= 5000  THEN '2 Low active (5,000 to 7,499)'
               ELSE                         '1 Sedentary (under 5,000)'
           END AS user_type,
           COUNT(*)                 AS users,
           ROUND(AVG(avg_steps))    AS avg_steps,
           ROUND(AVG(avg_calories)) AS avg_calories,
           ROUND(AVG(avg_sedentary_min)) AS avg_sedentary_min,
           ROUND(AVG(avg_mvpa_min), 1)   AS avg_mvpa_min,
           ROUND(AVG(days_logged), 1)    AS avg_days_logged
    FROM per_user
    GROUP BY user_type
    ORDER BY user_type
""")
user_types["share_of_users"] = user_types["users"] / user_types["users"].sum()
user_types

In [ ]:
per_user = sql("""
    SELECT Id,
           COUNT(*)              AS days_logged,
           AVG(TotalSteps)       AS avg_steps,
           AVG(Calories)         AS avg_calories,
           AVG(SedentaryMinutes) AS avg_sedentary_min,
           AVG(VeryActiveMinutes + FairlyActiveMinutes) AS avg_mvpa_min
    FROM daily_activity
    GROUP BY Id
""")

def step_group(x):
    if x >= 10000:
        return "Active"
    if x >= 7500:
        return "Somewhat active"
    if x >= 5000:
        return "Low active"
    return "Sedentary"

per_user["user_type"] = per_user["avg_steps"].apply(step_group)
type_order = ["Sedentary", "Low active", "Somewhat active", "Active"]
per_user["user_type"] = pd.Categorical(per_user["user_type"], categories=type_order, ordered=True)
per_user["user_type"].value_counts().sort_index()

### 4.3 How consistently do people wear the device?

In [ ]:
span_days = (daily_all["ActivityDate"].max() - daily_all["ActivityDate"].min()).days + 1
print("Length of the data window in days:", span_days)

per_user["share_of_days_logged"] = per_user["days_logged"] / span_days
per_user[["days_logged", "share_of_days_logged"]].describe().T[["mean", "min", "50%", "max"]]

In [ ]:
usage_band = pd.cut(
    per_user["days_logged"],
    bins=[0, 20, 40, 1000],
    labels=["Light use (up to 20 days)", "Moderate use (21 to 40 days)", "Heavy use (over 40 days)"],
)
usage_band.value_counts().sort_index()

### 4.4 Which relationships stand out?

In [ ]:
corr_cols = ["TotalSteps", "TotalDistance", "VeryActiveMinutes", "FairlyActiveMinutes",
             "LightlyActiveMinutes", "SedentaryMinutes", "Calories"]
corr = daily[corr_cols].corr()
corr["Calories"].drop("Calories").sort_values(ascending=False).to_frame("correlation with calories")

In [ ]:
slope, intercept = np.polyfit(daily["TotalSteps"], daily["Calories"], 1)
print(f"Each extra 1,000 steps goes with about {slope * 1000:,.0f} more calories burned")
print(f"Correlation between steps and calories: {daily['TotalSteps'].corr(daily['Calories']):.2f}")
print(f"Correlation between steps and sedentary minutes: {daily['TotalSteps'].corr(daily['SedentaryMinutes']):.2f}")

### 4.5 Which days of the week are busiest?

In [ ]:
weekday_summary = sql("""
    SELECT CASE strftime('%w', ActivityDate)
               WHEN '0' THEN 'Sunday'    WHEN '1' THEN 'Monday'
               WHEN '2' THEN 'Tuesday'   WHEN '3' THEN 'Wednesday'
               WHEN '4' THEN 'Thursday'  WHEN '5' THEN 'Friday'
               WHEN '6' THEN 'Saturday'
           END AS weekday,
           strftime('%w', ActivityDate) AS dow,
           COUNT(*)                     AS user_days,
           ROUND(AVG(TotalSteps))       AS avg_steps,
           ROUND(AVG(Calories))         AS avg_calories,
           ROUND(AVG(VeryActiveMinutes + FairlyActiveMinutes), 1) AS avg_mvpa_min,
           ROUND(AVG(SedentaryMinutes)) AS avg_sedentary_min
    FROM daily_activity
    GROUP BY dow
""")
order = {"1": 0, "2": 1, "3": 2, "4": 3, "5": 4, "6": 5, "0": 6}
weekday_summary["sort"] = weekday_summary["dow"].map(order)
weekday_summary = weekday_summary.sort_values("sort").drop(columns=["dow", "sort"]).reset_index(drop=True)
weekday_summary

### 4.6 At what time of day do people move?

In [ ]:
hourly_summary = sql("""
    SELECT CAST(strftime('%H', ActivityHour) AS INTEGER) AS hour,
           ROUND(AVG(StepTotal), 1) AS avg_steps,
           COUNT(*)                 AS user_hours
    FROM hourly_steps
    GROUP BY hour
    ORDER BY hour
""")
hourly_summary.sort_values("avg_steps", ascending=False).head(5)

In [ ]:
hourly["DayType"] = np.where(hourly["Weekday"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")
hourly_daytype = hourly.groupby(["DayType", "Hour"], as_index=False)["StepTotal"].mean()
hourly_daytype.pivot(index="Hour", columns="DayType", values="StepTotal").round(0).T

### 4.7 What does sleep look like, and does activity affect it?

In [ ]:
sleep_by_type = daily_sleep.merge(per_user[["Id", "user_type"]], on="Id")
sleep_by_type_summary = (
    sleep_by_type.groupby("user_type", observed=True)
    .agg(
        users=("Id", "nunique"),
        nights=("Id", "size"),
        avg_hours_asleep=("HoursAsleep", "mean"),
        avg_minutes_awake_in_bed=("MinutesAwakeInBed", "mean"),
        avg_sleep_efficiency=("SleepEfficiency", "mean"),
    )
)
sleep_by_type_summary

In [ ]:
sleep_sql_result = sql("""
    SELECT CASE
               WHEN s.TotalMinutesAsleep < 360 THEN '1 Under 6 hours'
               WHEN s.TotalMinutesAsleep < 420 THEN '2 6 to 7 hours'
               WHEN s.TotalMinutesAsleep < 480 THEN '3 7 to 8 hours'
               ELSE                                 '4 8 hours or more'
           END AS sleep_band,
           COUNT(*)                       AS nights,
           ROUND(AVG(a.TotalSteps))       AS avg_steps_same_day,
           ROUND(AVG(a.SedentaryMinutes)) AS avg_sedentary_min_same_day
    FROM sleep_day s
    JOIN daily_activity a
      ON a.Id = s.Id AND a.ActivityDate = s.SleepDay
    GROUP BY sleep_band
    ORDER BY sleep_band
""")
sleep_sql_result

In [ ]:
r_sleep_steps = daily_sleep["TotalSteps"].corr(daily_sleep["TotalMinutesAsleep"])
r_sleep_sedentary = daily_sleep["SedentaryMinutes"].corr(daily_sleep["TotalMinutesAsleep"])
r_bed_asleep = daily_sleep["TotalTimeInBed"].corr(daily_sleep["TotalMinutesAsleep"])
print(f"Steps vs minutes asleep:          r = {r_sleep_steps:.2f}")
print(f"Sedentary minutes vs minutes asleep: r = {r_sleep_sedentary:.2f}")
print(f"Time in bed vs minutes asleep:    r = {r_bed_asleep:.2f}")

short_sleep = (sleep["TotalMinutesAsleep"] < 420).mean()
print(f"Share of nights under 7 hours of sleep: {short_sleep:.1%}")

## 5. Share

The charts below tell the story from the data. Each one is saved to the `visuals` folder.

In [ ]:
PRIMARY = "#2F6F73"
ACCENT = "#E07A5F"
MUTED = "#B8C4C2"

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join("visuals", name), dpi=160, bbox_inches="tight")
    plt.show()

### Chart 1: Who is in the data

In [ ]:
counts = per_user["user_type"].value_counts().reindex(type_order)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(counts.index.astype(str), counts.values, color=[PRIMARY if v == counts.max() else MUTED for v in counts.values])
ax.bar_label(bars, padding=3)
ax.set_title("Users grouped by average daily steps", loc="left", fontweight="bold")
ax.set_xlabel("User type by average daily steps")
ax.set_ylabel("Number of users")
ax.set_ylim(0, counts.max() * 1.2)
finish(fig, "01_user_types.png")

### Chart 2: Steps and calories

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(
    data=daily, x="TotalSteps", y="Calories",
    scatter_kws={"alpha": 0.35, "s": 18, "color": PRIMARY},
    line_kws={"color": ACCENT, "linewidth": 2}, ax=ax,
)
ax.set_title("More steps go with more calories burned", loc="left", fontweight="bold")
ax.set_xlabel("Total steps in a day")
ax.set_ylabel("Calories burned")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:,.0f}")
finish(fig, "02_steps_vs_calories.png")

### Chart 3: Where the day goes

In [ ]:
minutes = daily[["SedentaryMinutes", "LightlyActiveMinutes", "FairlyActiveMinutes", "VeryActiveMinutes"]].mean()
labels = ["Sedentary", "Lightly active", "Fairly active", "Very active"]
share = minutes / minutes.sum()

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(labels[::-1], (share.values * 100)[::-1], color=[PRIMARY, MUTED, MUTED, ACCENT])
ax.bar_label(bars, labels=[f"{v:.1%}" for v in share.values[::-1]], padding=4)
ax.set_title("Most tracked time is spent sitting or resting", loc="left", fontweight="bold")
ax.set_xlabel("Share of tracked minutes (%)")
ax.set_xlim(0, share.max() * 100 * 1.15)
finish(fig, "03_time_by_intensity.png")

### Chart 4: Weekday pattern

In [ ]:
by_weekday = daily.groupby("Weekday", observed=True)["TotalSteps"].mean()
top_day = by_weekday.idxmax()

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = [ACCENT if d == top_day else MUTED for d in by_weekday.index]
bars = ax.bar(by_weekday.index.astype(str), by_weekday.values, color=colors)
ax.bar_label(bars, labels=[f"{v:,.0f}" for v in by_weekday.values], padding=3)
ax.axhline(daily["TotalSteps"].mean(), color=PRIMARY, linestyle="--", linewidth=1.2)
ax.text(6.45, daily["TotalSteps"].mean() + 80, "overall average", ha="right", color=PRIMARY, fontsize=9)
ax.set_title("Average steps by day of the week", loc="left", fontweight="bold")
ax.set_ylabel("Average steps")
ax.set_ylim(0, by_weekday.max() * 1.18)
finish(fig, "04_steps_by_weekday.png")

### Chart 5: Time of day pattern

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for day_type, color in [("Weekday", PRIMARY), ("Weekend", ACCENT)]:
    part = hourly_daytype[hourly_daytype["DayType"] == day_type]
    ax.plot(part["Hour"], part["StepTotal"], label=day_type, color=color, linewidth=2.2)
ax.set_title("Activity peaks around midday and early evening", loc="left", fontweight="bold")
ax.set_xlabel("Hour of the day")
ax.set_ylabel("Average steps in the hour")
ax.set_xticks(range(0, 24, 2))
ax.legend(frameon=False)
finish(fig, "05_steps_by_hour.png")

### Chart 6: Sleep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.scatterplot(
    data=sleep, x="TotalTimeInBed", y="TotalMinutesAsleep",
    alpha=0.5, color=PRIMARY, s=28, ax=axes[0],
)
lim = [0, sleep["TotalTimeInBed"].max() * 1.03]
axes[0].plot(lim, lim, color=ACCENT, linestyle="--", linewidth=1.2)
axes[0].set_title("Time in bed vs time asleep", loc="left", fontweight="bold")
axes[0].set_xlabel("Minutes in bed")
axes[0].set_ylabel("Minutes asleep")

axes[1].hist(sleep["HoursAsleep"], bins=18, color=MUTED, edgecolor="white")
axes[1].axvline(7, color=ACCENT, linestyle="--", linewidth=1.5)
axes[1].text(7.1, axes[1].get_ylim()[1] * 0.92, "7 hours", color=ACCENT)
axes[1].set_title("Nightly sleep duration", loc="left", fontweight="bold")
axes[1].set_xlabel("Hours asleep")
axes[1].set_ylabel("Number of nights")
finish(fig, "06_sleep.png")

### Chart 7: Feature usage and wear time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1, 1.1]})

usage = feature_usage.set_index("feature").loc[["Daily activity", "Sleep", "Weight"], "users"]
bars = axes[0].bar(usage.index, usage.values, color=[PRIMARY, MUTED, ACCENT])
axes[0].bar_label(bars, padding=3)
axes[0].set_title("Users per tracked feature", loc="left", fontweight="bold")
axes[0].set_ylabel("Number of users")
axes[0].set_ylim(0, usage.max() * 1.18)

axes[1].hist(per_user["days_logged"], bins=10, color=MUTED, edgecolor="white")
axes[1].set_title("Days of data per user", loc="left", fontweight="bold")
axes[1].set_xlabel("Days with tracked activity")
axes[1].set_ylabel("Number of users")
finish(fig, "07_feature_usage.png")

### Chart 8: Sleep by user type

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
data = sleep_by_type_summary["avg_hours_asleep"]
bars = ax.bar(data.index.astype(str), data.values, color=MUTED)
ax.bar_label(bars, labels=[f"{v:.1f} h" for v in data.values], padding=3)
ax.axhline(7, color=ACCENT, linestyle="--", linewidth=1.2)
ax.set_title("Average hours asleep by user type", loc="left", fontweight="bold")
ax.set_ylabel("Hours asleep")
ax.set_ylim(0, max(8, data.max() * 1.2))
finish(fig, "08_sleep_by_user_type.png")

### Key numbers from this run

The cell below pulls the headline numbers straight from the data so the written findings stay in line with the results.

In [ ]:
by_weekday = daily.groupby("Weekday", observed=True)["TotalSteps"].mean()
hour_peak = hourly_summary.sort_values("avg_steps", ascending=False).head(3)["hour"].tolist()
mins = daily[["SedentaryMinutes", "LightlyActiveMinutes", "FairlyActiveMinutes", "VeryActiveMinutes"]].mean()

findings = {
    "Users in activity data": users_daily,
    "Users with sleep data": users_sleep,
    "Users with weight data": weight["Id"].nunique(),
    "Average daily steps": round(daily["TotalSteps"].mean()),
    "Average calories per day": round(daily["Calories"].mean()),
    "Average sedentary hours per day": round(mins["SedentaryMinutes"] / 60, 1),
    "Share of tracked time sedentary": f"{mins['SedentaryMinutes'] / mins.sum():.1%}",
    "Average minutes very or fairly active": round(mins["VeryActiveMinutes"] + mins["FairlyActiveMinutes"], 1),
    "Most active weekday (steps)": f"{by_weekday.idxmax()} ({by_weekday.max():,.0f})",
    "Least active weekday (steps)": f"{by_weekday.idxmin()} ({by_weekday.min():,.0f})",
    "Busiest hours of the day": ", ".join(f"{h}:00" for h in hour_peak),
    "Average hours asleep": round(sleep["HoursAsleep"].mean(), 1),
    "Nights under 7 hours": f"{short_sleep:.1%}",
    "Average minutes awake in bed": round(sleep["MinutesAwakeInBed"].mean()),
    "Users reaching 10,000 steps on average": int((per_user["avg_steps"] >= 10000).sum()),
    "Users logging weight": weight["Id"].nunique(),
    "Weight entries entered manually": f"{weight['IsManualReport'].mean():.1%}",
}
pd.Series(findings, name="value").to_frame()

## 6. Act

### Which Bellabeat product?

I chose the **Bellabeat Time** wellness watch, together with the Bellabeat app. Time tracks activity, sleep and stress, so it covers the same ground as the Fitbit data I analyzed, and a watch is worn all day, which fits the patterns I found. The app is the place where every insight, nudge and membership offer reaches the customer.

### What the data says

1. **People wear their trackers for ordinary life, not for workouts.** Most tracked time is sedentary or lightly active, and only a small part of the day is spent in fairly or very active minutes. The device is a passive companion for the whole day.
2. **Activity follows a daily rhythm.** Movement rises in the morning, peaks around midday and in the early evening, and drops at night. Weekdays and weekends have different shapes, and the busiest weekday is not the same as the quietest one.
3. **Steps and calories move together, but steps alone miss intensity.** The link between steps and calories is strong, yet the sedentary share of the day stays large across the whole group.
4. **Sleep is tracked by fewer people than activity, and a large share of nights are shorter than seven hours.** There is also a gap between time in bed and time asleep, which is a clear coaching opportunity.
5. **Optional manual features are barely used.** Weight is logged by only a handful of users and mostly by hand, which suggests that anything that needs manual effort loses people fast.
6. **Wear time is uneven.** Some users have data for almost every day, others only for a short stretch, so engagement is the first thing to protect.

### Top three recommendations

**1. Make the Time watch the all day companion, with nudges tied to the daily rhythm.**
Use the hourly pattern to send gentle prompts at the moments when people are already moving, such as a midday walk reminder or an evening wind down. Add inactivity reminders aimed at the long sedentary stretches, and celebrate small wins like moving more minutes instead of only hitting a step goal.

**2. Turn sleep into a headline feature of the app and the membership.**
Fewer users track sleep than activity, and many nights are short. Marketing can focus on sleep quality, bedtime consistency and the gap between time in bed and time asleep. The Bellabeat membership can offer personalized sleep guidance, which gives people a reason to wear the watch at night too.

**3. Reduce manual effort and reward consistent wear.**
Because manual logging is rare, automate what can be automated (for example syncing with a smart scale) and use streaks, badges and weekly summaries to keep people wearing the watch every day. Campaigns can target the users who drop off early with simple onboarding goals for the first two weeks.

### Next steps and ideas for more data

* Validate the findings with Bellabeat's own customer data, which is current and covers stress, menstrual cycle and hydration.
* Compare usage between members and non members of the Bellabeat membership.
* Run a small test of the hourly nudges and measure how wear time and daily steps change.
* Look for a larger and more recent public dataset to test whether these patterns still hold.

### Limits of this analysis

The sample is small, the data is from 2016, there is no information on gender or age, and the people who agreed to share their data are probably more fitness minded than the average customer. The recommendations are directional and should be tested before a large investment.